# NYSC Survival · Nigerian voices with Gemini TTS (free tier)

1. Get a free API key at **https://aistudio.google.com/apikey** (sign in with Google → *Create API key*).
2. Run the cells **from top to bottom**. Step 3 makes 6 sample lines so you can judge the voices **before** spending your daily quota.
3. Step 4 generates the whole game. The free tier has a daily limit, so it saves progress to your Google Drive and you simply **run it again the next day** until it says *All done*.
4. Step 5 downloads `nysc_voices_gemini.zip` → send it to Claude.

In [ ]:
# 1 · setup (no installs needed: talks to Gemini directly, so library versions can't break it)
import os, json, time, wave, re, io, base64, subprocess, numpy as np, requests
print('ok')

In [ ]:
# 2 · your key + where to save (Google Drive keeps progress between days)
from getpass import getpass
API_KEY = getpass('Paste your Gemini API key: ').strip()
BASE = 'https://generativelanguage.googleapis.com/v1beta'
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nysc_voices_gemini'; os.makedirs(OUT, exist_ok=True)
# the line list comes straight from the game repo
SCRIPT_URL = 'https://raw.githubusercontent.com/Auspicious14/nysc-survival/main/game3d-src/voice/script.json'
LINES = requests.get(SCRIPT_URL, timeout=60).json()
done = {f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3')}
print(len(LINES), 'lines in the game ·', len(done), 'already done')
# find a TTS model this key can use
r = requests.get(f'{BASE}/models', params={'key': API_KEY, 'pageSize': 200}, timeout=60); r.raise_for_status()
names = [m['name'].split('/')[-1] for m in r.json().get('models', []) if 'tts' in m['name']]
MODEL = next((n for n in ['gemini-2.5-flash-preview-tts', 'gemini-2.5-flash-tts', 'gemini-2.5-pro-preview-tts'] if n in names), names[0] if names else 'gemini-2.5-flash-preview-tts')
print('TTS models:', names, '→ using', MODEL)

In [ ]:
# helpers: one request → PCM → mp3; batches split on long silences
def tts(prompt, voice):
    body = {'contents': [{'parts': [{'text': prompt}]}],
            'generationConfig': {'responseModalities': ['AUDIO'],
                                 'speechConfig': {'voiceConfig': {'prebuiltVoiceConfig': {'voiceName': voice}}}}}
    for attempt in range(4):
        try:
            r = requests.post(f'{BASE}/models/{MODEL}:generateContent', params={'key': API_KEY}, json=body, timeout=180)
        except Exception as e:
            print('   network:', str(e)[:120]); time.sleep(8); continue
        if r.status_code == 200:
            part = r.json()['candidates'][0]['content']['parts'][0]
            return np.frombuffer(base64.b64decode(part['inlineData']['data']), dtype=np.int16)
        msg = r.text
        if r.status_code == 429:
            if 'PerDay' in msg or 'per day' in msg.lower() or attempt == 3: raise RuntimeError('DAILY_LIMIT')
            print('   rate limit, waiting 65 s…'); time.sleep(65); continue
        print(f'   error {r.status_code}:', msg[:400]); time.sleep(8)
    raise RuntimeError('FAILED')
def save_mp3(pcm, path):
    wav = path[:-4] + '.wav'
    with wave.open(wav, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(24000); w.writeframes(pcm.tobytes())
    subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', wav, '-af', 'silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse,apad=pad_dur=0.12,loudnorm=I=-16:TP=-1.5', '-ar', '24000', '-ac', '1', '-b:a', '96k', path], check=True)
    os.remove(wav)
def split_silence(pcm, n, min_sil=0.75):
    fr = 480  # 20 ms
    k = len(pcm) // fr; rms = np.sqrt((pcm[:k * fr].astype(np.float32).reshape(k, fr) ** 2).mean(1))
    quiet = rms < max(220, np.percentile(rms, 20) * 1.6)
    runs, s = [], None
    for i, q in enumerate(quiet):
        if q and s is None: s = i
        if not q and s is not None:
            if (i - s) * 0.02 >= min_sil and s > 0: runs.append((s, i))
            s = None
    runs = sorted(runs, key=lambda r: r[1] - r[0], reverse=True)[:n - 1]
    cuts = sorted(((a + b) // 2) * fr for a, b in runs)
    if len(cuts) != n - 1: return None
    return np.split(pcm, cuts)
def prompt_for(group):
    style = group[0]['style']
    if len(group) == 1: return f"{style}\n\n{group[0]['text']}"
    body = '\n'.join(f"{i + 1}. {l['text']}" for i, l in enumerate(group))
    return f"{style}\nRead these {len(group)} separate lines one after another. Do NOT read the numbers. After each line stay completely silent for two full seconds.\n\n{body}"
print('ready')

In [ ]:
# 3 · SAMPLES (optional — skip once you like the voices). Samples are saved as real game lines, so no quota is wasted.
import glob, shutil
from IPython.display import Audio, display
for s in glob.glob('/content/sample_*.mp3'):  # keep samples from an earlier run
    shutil.move(s, f"{OUT}/{os.path.basename(s)[7:]}")
have = {f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3')}
for want in ['Fenrir', 'Charon', 'Kore', 'Puck', 'Sulafat', 'Orus']:
    l = next((x for x in LINES if x['voice'] == want and x['prio'] == 1), None)
    if not l: continue
    p = f"{OUT}/{l['id']}.mp3"
    if l['id'] not in have: save_mp3(tts(prompt_for([l]), l['voice']), p)
    print(l['voice'], '·', l['text']); display(Audio(p))


In [ ]:
# 4 · GENERATE EVERYTHING (run again tomorrow if it stops at the daily limit)
BATCH = 8
todo = [l for l in LINES if l['id'] not in {f[:-4] for f in os.listdir(OUT)}]
print(len(todo), 'lines to go')
groups, cur = [], []
for l in todo:  # same voice + same style per request
    if cur and (cur[0]['voice'] != l['voice'] or cur[0]['style'] != l['style'] or len(cur) >= BATCH): groups.append(cur); cur = []
    cur.append(l)
if cur: groups.append(cur)
made = 0
try:
    for gi, g in enumerate(groups):
        pcm = tts(prompt_for(g), g[0]['voice'])
        parts = [pcm] if len(g) == 1 else split_silence(pcm, len(g))
        if parts is None:  # could not split cleanly → try once more, then halves
            pcm = tts(prompt_for(g), g[0]['voice']); parts = split_silence(pcm, len(g))
        if parts is None:
            h = len(g) // 2; parts = []
            for sub in (g[:h], g[h:]):
                p2 = tts(prompt_for(sub), sub[0]['voice']); sp = [p2] if len(sub) == 1 else split_silence(p2, len(sub))
                parts += sp if sp else [tts(prompt_for([l]), l['voice']) for l in sub]
        for l, part in zip(g, parts): save_mp3(part, f"{OUT}/{l['id']}.mp3"); made += 1
        print(f"{gi + 1}/{len(groups)} · {g[0]['voice']} · {made} new files")
        time.sleep(7)  # stay under the per-minute limit
    print('All done ✅')
except RuntimeError as e:
    print(f"Stopped: {e}. {made} new files saved to Drive. Run this cell again tomorrow to continue.")

In [ ]:
# 5 · download the zip for Claude (works with whatever is finished so far)
import shutil
ids = sorted(f[:-4] for f in os.listdir(OUT) if f.endswith('.mp3'))
json.dump(ids, open(f'{OUT}/manifest.json', 'w'))
shutil.make_archive('/content/nysc_voices_gemini', 'zip', OUT)
print(len(ids), 'voice files zipped')
from google.colab import files; files.download('/content/nysc_voices_gemini.zip')